<a href="https://colab.research.google.com/github/Matheus4ugusto/card-2-fastcamp-LAMIA/blob/main/pratica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [215]:
%pip install groq

In [216]:
import os
os.environ['GROQ_API_KEY'] = ""
os.environ['NINJA_API_KEY'] = ""
from groq import Groq
import requests
import re

In [217]:
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [218]:
class Agent:
  def __init__(self, client: Groq, system: str = "") -> None:
    self.client = client
    self.system = system
    self.messages: list = []
    if self.system:
      self.messages.append({"role": "system", "content": system})

  def __call__(self, message=""):
    if message:
      self.messages.append({"role": "user", "content": message})
      result = self.execute()
      self.messages.append({"role": "assistant", "content": result})
      return result
    else:
      return self.execute()

  def execute(self):
    completion = client.chat.completions.create(
        model="llama-3.1-8b-instant", messages=self.messages, tool_choice="none"
    )
    return completion.choices[0].message.content

In [219]:
system_prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer.

Use Thought to describe your reasoning about the question you have been asked.
Use Action to run one of the actions available to you — then return PAUSE.
Observation will be the result of running those actions.

IMPORTANT RULE:
If you need the population of any country, you MUST call the `get_population` action.
Never guess or estimate population values yourself. Always use the tool to retrieve the correct data.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number. Uses Python syntax, so use floating point when necessary.

get_population:
e.g. get_population: Brazil
Returns the population of the specified country.

Example session:

Question: What is double the population of Brazil?

Thought: I need to find the population of Brazil, and according to the rules I must use the tool to retrieve it.
Action: get_population: Brazil
PAUSE

You will be called again with this:

Observation: 203062512

Thought: I need to multiply this value by 2.
Action: calculate: 203062512 * 2
PAUSE

You will be called again with this:

Observation: 406125024

Thought: I now have the final result.

Answer: Double the population of Brazil is 406125024.

Now it's your turn:
""".strip()

In [220]:
# tools

def get_population(country):
    url = "https://api.api-ninjas.com/v1/population"

    headers = {
        "X-Api-Key": os.environ["NINJA_API_KEY"]
    }

    params = {
        "country": country
    }

    response = requests.get(url, headers=headers, params=params)
    return response.json()["historical_population"][0]["population"]

def calculate(operation):
    return eval(operation)

def get_max(a, b):
    return max(a, b)

In [221]:
def loop(max_iterations=100, query: str = ""):
  PopulationAgent = Agent(client=client, system=system_prompt)

  tools = ["get_population", "calculate"]

  next_prompt = query

  i = 0

  while i < max_iterations:
    i += 1
    print(f"Iteration: {i}")
    result = PopulationAgent(next_prompt)
    print(result)

    if "PAUSE" in result and "Action" in result:
      action = re.findall(r"Action: ([a-z_]+): (.+)", result, re.IGNORECASE)
      chosen_tool = action[0][0]
      arg = action[0][1]

      if chosen_tool in tools:
        result_tool = eval(f"{chosen_tool}('{arg}')")
        next_prompt = f"Observation: {result_tool}"
      else:
        next_prompt = "Observation: Tool not found"

      print(next_prompt)
      continue

    if "Answer" in result:
      break

In [222]:
loop(query="Question: Among the countries China and India, wich one have the biggest population?")

Iteration: 1
Thought: I need to find out the population of both China and India, and according to the rules I must use the tool to retrieve it.
Action: get_population: China
PAUSE
Observation: 1419321278
Iteration: 2
Thought: I also need to find out the population of India to make a comparison.
Action: get_population: India
PAUSE
Observation: 1450935791
Iteration: 3
Thought: Now that I have the populations for both China and India, I need to compare them to find which one has the biggest population.
Action: calculate: 1450935791 > 1419321278
PAUSE
Observation: True
Iteration: 4
Thought: The calculation shows that the population of India is greater than the population of China, which means India has the biggest population.
Answer: India has the biggest population among the two countries.
